# CASMI26: Isomer Ranker Train

Fits the isomer joint ranker used by [CASMI26: Anchored Spectral Network](https://www.kaggle.com/code/flexonafft/casmi26-anchored-spectral-network)
from v3 on. For a molecule's formula it takes the candidate isomers from the PubChem tier, scores each by the fold-safe
FPNet fold 0, by its documentation and by its natural-product membership, and orders them with a LightGBM lambdarank
model. The overlay then lets that ranker place its first choice at rank 2 (or, above a stricter margin, at rank 1).

The idea is Heliosli's CV-JOINT overlay (FPNet + documentation + natural products -> a LightGBM ranker that re-judges
rank 1). His fitted files are no longer public, so this notebook fits our own version from public inputs only, without his
RT and PX v2 models, and publishes the whole fit: nothing here is trained elsewhere.

| | |
|---|---|
| training molecules | structures of identity fold 0, which FPNet fold 0 never saw; timsTOF / Enveda molecules first |
| candidates | the PubChem tier at the formula's exact mass, at most 1,000 structures, most documented first |
| features | 16 per candidate: FPNet f.z (z-score, gap, rank), documentation, channel score z(FPNet) + 0.25 x documentation, natural-product bits |
| model | LightGBM lambdarank, 400 rounds, five-fold out-of-fold report, then one fit on every molecule |
| thresholds | margins where the out-of-fold first choice is right 90% (rank 1) and 50% (rank 2) of the time, on test-like molecules |
| outputs | `isomer_ranker.py` (the shared feature code), `isomer_ranker_model.txt`, `isomer_ranker_meta.json`, the out-of-fold table |

Credits: Heliosli for the CV-JOINT design; Dmitrii Gluzdov for the fold-safe FPNet, the identity folds and the PubChem
popularity prior; Ahmed Berat Özer for the FPNet architecture, the fingerprint bits and the `casmi` code; Enveda for the
competition spectra. Licences of those inputs apply to the files derived from them.

In [ ]:
MODULE_SRC = '"""Isomer joint ranker (flexonafft, CASMI26).\n\nCandidate isomers of one molecular formula come from the PubChem tier of\ndmitriigluzdov/casmi26-pubchem-popularity-prior. Each candidate is scored by the fold-safe FPNet fold 0\n(dmitriigluzdov/casmi26-fold-safe-fpnet, f.z on the selected fingerprint bits), by its documentation\n(ln substance records + ln PubMed articles) and by its natural-product membership bits. A LightGBM lambdarank\nmodel fitted on training structures of identity fold 0 (never seen by FPNet fold 0) orders them.\n\nThe idea follows Heliosli\'s CV-JOINT overlay (FPNet + documentation + natural products -> LightGBM ranker, which\nre-judges rank 1). His fitted models are no longer public, so this module is our own re-implementation without\nthe RT and PX v2 models, fitted in the public notebook flexonafft/casmi26-isomer-ranker-train.\n\nOne module serves both the training notebook and the submission overlay, so the candidates and the features are\ncomputed by one code path.\n"""\nimport glob\nimport json\nimport os\nimport sys\nimport time\n\nimport numpy as np\n\nK1_CHAN, K1_FZ, K1_POP = 150, 50, 50     # stage 1: union of the best by channel score, by FPNet and by documentation\nCAP = 1000                               # at most this many tier structures per formula (most documented first)\nMASS_TOL = 2e-4                          # Da around the formula\'s exact mass; the formula itself is checked after parsing\nMAX_SPECTRA = 8\nFEATS = [\'fzz\', \'fz_gap\', \'lfz_rank\', \'pop\', \'pop_gap\', \'lpop_rank\', \'chan\', \'chan_gap\', \'lchan_rank\',\n         \'ls\', \'lp\', \'isnp\', \'np_coconut\', \'np_lotus\', \'np_npatlas\', \'ln_n\']\n\n\ndef find(inp, pattern):\n    hits = sorted(glob.glob(os.path.join(inp, \'**\', pattern), recursive=True), key=len)\n    return hits[0] if hits else None\n\n\ndef locate(inp=\'/kaggle/input\'):\n    P = {}\n    P[\'fpnet0\'] = find(inp, \'casmi26-fold-safe-fpnet/fpnet_fold0.pt\') or find(inp, \'fpnet_fold0.pt\')\n    P[\'bits\'] = os.path.join(os.path.dirname(P[\'fpnet0\']), \'fp_bits.npy\') if P[\'fpnet0\'] else None\n    lsid = find(inp, \'casmi26-pubchem-popularity-prior/pc_lsid.npy\') or find(inp, \'pc_lsid.npy\')\n    P[\'pop\'] = os.path.dirname(lsid) if lsid else None\n    P[\'tier\'] = os.path.join(P[\'pop\'], \'casmi26-pubchem-tier\') if P[\'pop\'] else None\n    fp_code = find(inp, \'casmi26-v3-models/code/casmi/fpnet.py\') or find(inp, \'casmi/fpnet.py\')\n    P[\'code\'] = os.path.dirname(os.path.dirname(fp_code)) if fp_code else None\n    missing = [k for k, v in P.items() if not v or not os.path.exists(v)]\n    if missing:\n        raise FileNotFoundError(\'isomer_ranker inputs missing: %s\' % missing)\n    if P[\'code\'] not in sys.path:\n        sys.path.insert(0, P[\'code\'])\n    return P\n\n\nclass Tier:\n    """Memory-mapped PubChem tier: rows sorted by neutral monoisotopic mass, all arrays row-aligned."""\n\n    def __init__(self, P):\n        t, r = P[\'tier\'], P[\'pop\']\n        self.M = np.load(os.path.join(t, \'pc_mass.npy\'), mmap_mode=\'r\')\n        self.O = np.load(os.path.join(t, \'pc_off.npy\'), mmap_mode=\'r\')\n        self.S = np.load(os.path.join(t, \'pc_smiles.npy\'), mmap_mode=\'r\')\n        self.K = np.load(os.path.join(r, \'pc_ik14.npy\'), mmap_mode=\'r\')\n        self.NP = np.load(os.path.join(r, \'pc_np.npy\'), mmap_mode=\'r\')\n        self.LS = np.load(os.path.join(r, \'pc_lsid.npy\'), mmap_mode=\'r\')\n        self.LP = np.load(os.path.join(r, \'pc_lpmid.npy\'), mmap_mode=\'r\')\n        n = len(self.M)\n        assert len(self.K) == len(self.NP) == len(self.LS) == len(self.LP) == n and len(self.O) == n + 1, \'tier arrays differ in length\'\n\n    def smiles(self, i):\n        return bytes(self.S[int(self.O[i]):int(self.O[i + 1])]).decode(\'ascii\', \'ignore\')\n\n    def window(self, m0, cap=CAP):\n        """Distinct structures (by stored InChIKey-14) within MASS_TOL of m0, most documented first, at most cap.\n        The choice never looks at a label, so training and the overlay see the same population."""\n        lo = int(np.searchsorted(self.M, m0 - MASS_TOL))\n        hi = int(np.searchsorted(self.M, m0 + MASS_TOL))\n        if hi <= lo:\n            return []\n        ls = np.nan_to_num(np.asarray(self.LS[lo:hi], np.float32))\n        lp = np.nan_to_num(np.asarray(self.LP[lo:hi], np.float32))\n        keys = np.asarray(self.K[lo:hi])\n        npb = np.asarray(self.NP[lo:hi])\n        out, seen = [], set()\n        for j in np.argsort(-(ls + lp), kind=\'stable\'):\n            k = bytes(keys[j]).decode(\'ascii\', \'ignore\')\n            if not k or k in seen:\n                continue\n            seen.add(k)\n            out.append((lo + int(j), k, float(ls[j]), float(lp[j]), int(npb[j])))\n            if len(out) >= cap:\n                break\n        return out\n\n\n# ---- fingerprints (pool workers) ----------------------------------------------------------------------------------\n_W = {}\n\n\ndef fp_init(code, bits_path):\n    if code not in sys.path:\n        sys.path.insert(0, code)\n    from casmi import chem\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.rdMolDescriptors import CalcMolFormula\n    RDLogger.DisableLog(\'rdApp.*\')\n    _W.update(chem=chem, Chem=Chem, cf=CalcMolFormula, bits=np.load(bits_path))\n\n\ndef fp_job(job):\n    """(smiles, formula) -> packed FPNet-bit fingerprint, or None when unparsable or of another formula."""\n    smi, f0 = job\n    try:\n        m = _W[\'Chem\'].MolFromSmiles(smi) if smi else None\n        if m is None or _W[\'cf\'](m) != f0:\n            return None\n        raw = _W[\'chem\'].raw_fingerprint(smi)\n        if raw is None:\n            return None\n        return np.packbits(raw[_W[\'bits\']].astype(np.uint8))\n    except Exception:\n        return None\n\n\ndef formula_mass(smiles):\n    """(formula, exact mass) of a SMILES with RDKit, or (None, None)."""\n    from rdkit import Chem\n    from rdkit.Chem import Descriptors\n    from rdkit.Chem.rdMolDescriptors import CalcMolFormula\n    try:\n        m = Chem.MolFromSmiles(smiles) if smiles else None\n        if m is None:\n            return None, None\n        return CalcMolFormula(m), float(Descriptors.ExactMolWt(m))\n    except Exception:\n        return None, None\n\n\n# ---- FPNet fold 0 logits per molecule -----------------------------------------------------------------------------\ndef _items(rs):\n    from casmi import chem, fpnet\n    from casmi.spectra import merge_spectra\n    sp = []\n    for s_ in rs[:MAX_SPECTRA]:\n        try:\n            mz = np.asarray(s_[\'ms2_mzs\'], np.float64)\n            it = np.asarray(s_[\'ms2_normalized_intensities\'], np.float64)\n        except Exception:\n            continue\n        if mz.size == 0 or mz.size != it.size:\n            continue\n        try:\n            arr = np.asarray(s_.get(\'collision_energy_ev\'), np.float64).ravel()\n            ok = np.isfinite(arr)\n            ce = float(arr[ok].mean()) if ok.any() else np.nan\n            ce_n = int(ok.sum()) if arr.size else 1\n        except Exception:\n            ce, ce_n = np.nan, 1\n        sp.append(dict(mz=mz, it=it, prec=float(s_[\'precursor_mz\']), adduct=str(s_[\'adduct\']), ce=ce, ce_n=max(1, ce_n),\n                       mode=1 if str(s_[\'ionization_mode\']) == \'positive\' else -1))\n    items_s = []\n    for s_ in sp:\n        pm, pi = fpnet.prep_peaks(s_[\'mz\'], s_[\'it\'], s_[\'prec\'])\n        ok = not np.isnan(s_[\'ce\'])\n        items_s.append(dict(mz=pm, it=pi, prec=s_[\'prec\'], adduct_ix=chem.adduct_index(s_[\'adduct\']), ce=s_[\'ce\'] if ok else 0.0,\n                            ce_known=1.0 if ok else 0.0, n_merged=min(s_[\'ce_n\'], 8), mode=float(s_[\'mode\'])))\n    items_m = []\n    for md in (1, -1):\n        grp = [s_ for s_ in sp if s_[\'mode\'] == md]\n        if not grp:\n            continue\n        mm, ii = merge_spectra([(s_[\'mz\'], s_[\'it\']) for s_ in grp])\n        prec = float(np.median([s_[\'prec\'] for s_ in grp]))\n        pm, pi = fpnet.prep_peaks(mm, ii, prec)\n        ces = [s_[\'ce\'] for s_ in grp if not np.isnan(s_[\'ce\'])]\n        ads = [s_[\'adduct\'] for s_ in grp]\n        items_m.append(dict(mz=pm, it=pi, prec=prec, adduct_ix=chem.adduct_index(max(set(ads), key=ads.count)),\n                            ce=float(np.mean(ces)) if ces else 0.0, ce_known=1.0 if ces else 0.0,\n                            n_merged=min(sum(s_[\'ce_n\'] for s_ in grp), 8), mode=float(md)))\n    return items_s, items_m\n\n\ndef fpnet_z(P, spectra_by_mol, device=None, chunk=2048):\n    """{molecule: list of spectrum dicts} -> {molecule: float32 logits}; z = 0.5 (mean single + mean merged)."""\n    import torch\n    from casmi import fpnet\n    dev = device or (\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    bank = fpnet.ModelBank([P[\'fpnet0\']], dev)\n    nb = len(np.load(P[\'bits\']))\n    assert bank.nbits == nb, (bank.nbits, nb)\n    flat, owner = [], []\n    for mid, rs in spectra_by_mol.items():\n        s, m = _items(rs)\n        if not s:\n            continue\n        flat += s + m\n        owner += [(mid, 0)] * len(s) + [(mid, 1)] * len(m)\n    acc = {}\n    for a in range(0, len(flat), chunk):\n        zz = bank.logits(flat[a:a + chunk], bs=128)\n        for (mid, kind), z in zip(owner[a:a + chunk], zz):\n            acc.setdefault(mid, ([], []))[kind].append(z)\n    out = {}\n    for mid, (s, m) in acc.items():\n        zs = np.mean(s, 0)\n        zm = np.mean(m, 0) if m else zs\n        out[mid] = (0.5 * (zs + zm)).astype(np.float32)\n    return out\n\n\n# ---- features -----------------------------------------------------------------------------------------------------\ndef _rank(v):\n    return np.argsort(np.argsort(-v, kind=\'stable\'), kind=\'stable\') + 1\n\n\ndef features(fz, ls, lp, npb):\n    """Stage-1 selection and the feature matrix of one molecule\'s candidates (no label is used)."""\n    n = len(fz)\n    fzz = (fz - fz.mean()) / (fz.std() + 1e-9)\n    pop = ls + lp\n    chan = fzz + 0.25 * pop\n    sel = set(np.argsort(-chan, kind=\'stable\')[:K1_CHAN].tolist())\n    sel |= set(np.argsort(-fz, kind=\'stable\')[:K1_FZ].tolist())\n    sel |= set(np.argsort(-pop, kind=\'stable\')[:K1_POP].tolist())\n    sel = np.array(sorted(sel), np.int64)\n    o_f, o_p, o_c = _rank(fz), _rank(pop), _rank(chan)\n    nb = npb[sel].astype(np.int64)\n    F = dict(fzz=fzz[sel], fz_gap=fzz[sel] - fzz.max(), lfz_rank=np.log(o_f[sel]),\n             pop=pop[sel], pop_gap=pop[sel] - pop.max(), lpop_rank=np.log(o_p[sel]),\n             chan=chan[sel], chan_gap=chan[sel] - chan.max(), lchan_rank=np.log(o_c[sel]),\n             ls=ls[sel], lp=lp[sel], isnp=(nb > 0), np_coconut=(nb & 1) > 0, np_lotus=(nb & 2) > 0, np_npatlas=(nb & 4) > 0,\n             ln_n=np.full(len(sel), np.log(n)))\n    X = np.stack([np.asarray(F[f], np.float64) for f in FEATS], 1).astype(np.float32)\n    return sel, X\n\n\ndef molecule_matrix(cands, fps, z, nbits):\n    """cands: Tier.window rows; fps: packed fingerprints (None = dropped). -> (kept candidates, X) or None."""\n    keep = [i for i, f in enumerate(fps) if f is not None]\n    if len(keep) < 2:\n        return None\n    cfp = np.unpackbits(np.stack([fps[i] for i in keep]), axis=1)[:, :nbits].astype(np.float32)\n    fz = cfp @ z.astype(np.float32)\n    ls = np.array([cands[i][2] for i in keep], np.float32)\n    lp = np.array([cands[i][3] for i in keep], np.float32)\n    npb = np.array([cands[i][4] for i in keep], np.int64)\n    sel, X = features(fz.astype(np.float64), ls.astype(np.float64), lp.astype(np.float64), npb)\n    return [cands[keep[j]] for j in sel], X, fz[sel]\n\n\ndef fingerprints(tier, jobs_by_formula, pool, cache):\n    """Fill cache[(row, formula)] for every (row, formula) job not yet cached, with one pool.map."""\n    todo = [(r, f) for f, rows in jobs_by_formula.items() for r in rows if (r, f) not in cache]\n    if not todo:\n        return\n    res = pool.map(fp_job, [(tier.smiles(r), f) for r, f in todo], chunksize=64)\n    for k, v in zip(todo, res):\n        cache[k] = v\n\n\n# ---- the submission overlay ---------------------------------------------------------------------------------------\ndef _keys(smi, cache):\n    """{raw InChIKey-14, scorer key} of a SMILES (either one matching counts as the same structure)."""\n    if smi in cache:\n        return cache[smi]\n    out = set()\n    try:\n        from rdkit import Chem\n        from casmi import chem\n        m = Chem.MolFromSmiles(smi)\n        if m is not None:\n            k = Chem.MolToInchiKey(m)\n            if k:\n                out.add(k[:14])\n        sk = chem.score_key(smi)\n        if sk:\n            out.add(sk)\n    except Exception:\n        pass\n    cache[smi] = out\n    return out\n\n\ndef apply_main(argv=None):\n    import argparse\n    import multiprocessing as mp\n    import pandas as pd\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--work\', default=\'/kaggle/working\')\n    ap.add_argument(\'--inp\', default=\'/kaggle/input\')\n    ap.add_argument(\'--test\', required=True)\n    ap.add_argument(\'--model_dir\', required=True)\n    ap.add_argument(\'--mode\', default=\'insert\', choices=[\'insert\', \'swap\', \'audit\'])\n    ap.add_argument(\'--swap\', type=float, default=None)\n    ap.add_argument(\'--ins\', type=float, default=None)\n    ap.add_argument(\'--lib_tau\', type=float, default=0.9)\n    ap.add_argument(\'--seconds\', type=float, default=3000)\n    a = ap.parse_args(argv)\n    t0 = time.monotonic()\n    import lightgbm as lgb\n    P = locate(a.inp)\n    meta = json.load(open(os.path.join(a.model_dir, \'isomer_ranker_meta.json\')))\n    assert meta[\'features\'] == FEATS, \'feature list differs from the fitted model\'\n    booster = lgb.Booster(model_file=os.path.join(a.model_dir, \'isomer_ranker_model.txt\'))\n    t_swap = a.swap if a.swap is not None else meta[\'thresholds\'].get(\'swap\')\n    t_ins = a.ins if a.ins is not None else meta[\'thresholds\'].get(\'insert\')\n    sub_path = os.path.join(a.work, \'submission.csv\')\n    sub = pd.read_csv(sub_path, dtype=str, keep_default_na=False)\n    lib = {}\n    ev_path = os.path.join(a.work, \'frozen_candidate_evidence.parquet\')\n    if os.path.exists(ev_path):\n        ev = pd.read_parquet(ev_path, columns=[\'molecule_id\', \'smiles\', \'library_max\'])\n        for m, s, x in zip(ev.molecule_id.astype(str), ev.smiles.astype(str), ev.library_max):\n            if x == x:\n                lib[(m, s)] = max(float(x), lib.get((m, s), 0.0))\n                lib[m] = max(float(x), lib.get(m, 0.0))\n    te = pd.read_parquet(a.test)\n    spectra = {}\n    for r in te.to_dict(\'records\'):\n        spectra.setdefault(str(r[\'molecule_id\']), []).append(r)\n    audit = dict(mode=a.mode, swap=t_swap, ins=t_ins, molecules=len(sub), with_list=0, scored=0, agree=0, swapped=0,\n                 inserted=0, below=0, library_kept=0, already_rank2=0, no_candidates=0, failed=0, timed_out=0,\n                 lib_source=\'evidence\' if lib else \'none\')\n    jobs, lists = {}, {}\n    tier = Tier(P)\n    for mid, s in zip(sub.molecule_id.astype(str), sub.smiles.astype(str)):\n        L = [x for x in s.split(\';\') if x]\n        if len(L) < 2 or mid not in spectra:\n            continue\n        f0, m0 = formula_mass(L[0])\n        if f0 is None:\n            continue\n        cands = tier.window(m0)\n        if not cands:\n            audit[\'no_candidates\'] += 1\n            continue\n        lists[mid] = (L, f0, cands)\n        jobs.setdefault(f0, set()).update(c[0] for c in cands)\n    audit[\'with_list\'] = len(lists)\n    cache = {}\n    with mp.get_context(\'fork\').Pool(max(1, (os.cpu_count() or 2)), initializer=fp_init, initargs=(P[\'code\'], P[\'bits\'])) as pool:\n        fingerprints(tier, jobs, pool, cache)\n    audit[\'fp_seconds\'] = round(time.monotonic() - t0, 1)\n    Z = fpnet_z(P, {m: spectra[m] for m in lists})\n    nbits = len(np.load(P[\'bits\']))\n    kc, new, dump = {}, {}, {}\n    for mid, (L, f0, cands) in lists.items():\n        if time.monotonic() - t0 > a.seconds:\n            audit[\'timed_out\'] += 1\n            continue\n        try:\n            if mid not in Z:\n                continue\n            mm = molecule_matrix(cands, [cache.get((c[0], f0)) for c in cands], Z[mid], nbits)\n            if mm is None:\n                audit[\'no_candidates\'] += 1\n                continue\n            sc, X, _ = mm\n            js = booster.predict(X, num_iteration=meta.get(\'best_iteration\'))\n            o = np.argsort(-js, kind=\'stable\')\n            best = sc[o[0]]\n            margin = float(js[o[0]] - js[o[1]]) if len(o) > 1 else 9.0\n            audit[\'scored\'] += 1\n            keys = [_keys(x, kc) for x in L]\n            hit = next((i for i, ks in enumerate(keys) if best[1] in ks), None)\n            if hit == 0:\n                act = \'agree\'\n            else:\n                best_smi = L[hit] if hit is not None else tier.smiles(best[0])\n                rest = [x for i, x in enumerate(L) if i != hit and i != 0]\n                lb = lib.get((mid, L[0]), lib.get(mid, 0.0))\n                strong = t_swap is not None and margin >= t_swap\n                can_swap = a.mode == \'swap\' and strong and lb < a.lib_tau\n                can_ins = a.mode in (\'swap\', \'insert\') and t_ins is not None and margin >= t_ins\n                if can_swap:\n                    new[mid] = ([best_smi, L[0]] + rest)[:25]\n                    act = \'swapped\'\n                elif can_ins and hit != 1:\n                    new[mid] = ([L[0], best_smi] + rest)[:25]\n                    act = \'inserted\'\n                elif can_ins:\n                    act = \'already_rank2\'\n                else:\n                    act = \'below\'\n                if a.mode == \'swap\' and strong and lb >= a.lib_tau:\n                    audit[\'library_kept\'] += 1\n            audit[act] = audit.get(act, 0) + 1\n            dump[mid] = dict(act=act, best_key=best[1], margin=round(margin, 4), hit=hit, n=len(sc), formula=f0)\n        except Exception as ex:\n            audit[\'failed\'] += 1\n            audit[\'last_error\'] = repr(ex)[:300]\n    if a.mode != \'audit\' and new:\n        out = [\';\'.join(new.get(m) or [x for x in s.split(\';\') if x]) for m, s in zip(sub.molecule_id.astype(str), sub.smiles.astype(str))]\n        res = sub.copy()\n        res[\'smiles\'] = out\n        assert len(res) == len(sub) and list(res.molecule_id) == list(sub.molecule_id)\n        lens = res.smiles.str.split(\';\').map(len)\n        assert lens.max() <= 25 and all(len(set(x.split(\';\'))) == len(x.split(\';\')) for x in res.smiles)\n        tmp = os.path.join(a.work, \'submission.isomer.tmp.csv\')\n        res.to_csv(tmp, index=False)\n        os.replace(tmp, sub_path)\n    audit[\'seconds\'] = round(time.monotonic() - t0, 1)\n    json.dump(audit, open(os.path.join(a.work, \'isomer_ranker_audit.json\'), \'w\'), indent=1)\n    json.dump(dump, open(os.path.join(a.work, \'isomer_ranker_decisions.json\'), \'w\'))\n    print(\'[isomer] done:\', json.dumps(audit), flush=True)\n\n\nif __name__ == \'__main__\':\n    apply_main()\n'

# Session: the scorer's RDKit, the shared ranker module, the mounted inputs.
import glob, importlib.metadata, json, os, subprocess, sys, time
from pathlib import Path

T0 = time.time()
os.environ['PYTHONHASHSEED'] = '0'
try:
    _installed = importlib.metadata.version('rdkit')
except importlib.metadata.PackageNotFoundError:
    _installed = ''
if _installed.replace('.03.', '.3.') != '2026.3.3':   # the same wheel and check as the 0.465 pipeline's bootstrap
    _tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
    _wheels = sorted(Path('/kaggle/input').rglob(f'rdkit-2026.3.3-{_tag}-*.whl'))
    assert _wheels, f'Attach the RDKit 2026.3.3 offline wheel for {_tag}.'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '-q', str(_wheels[0])],
                          env={**os.environ, 'PYTHONHASHSEED': '0'})
from rdkit import rdBase
assert rdBase.rdkitVersion == '2026.03.3', rdBase.rdkitVersion

WORK = Path('/kaggle/working')
(WORK / 'isomer_ranker.py').write_text(MODULE_SRC)
sys.path.insert(0, str(WORK))
import isomer_ranker as IR

P = IR.locate('/kaggle/input')
COMP = next(p for p in (Path('/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra'),
                        Path('/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra'),
                        Path('/kaggle/input/enveda-casmi26-molecule-id-mass-spectra')) if (p / 'train.parquet').is_file())
FOLDS = IR.find('/kaggle/input', 'casmi26-identity-disjoint-validation-folds/structures.parquet')
assert FOLDS, 'Attach dmitriigluzdov/casmi26-natural-product-panel-benchmark (identity folds)'
print('RDKit', rdBase.rdkitVersion, '| inputs', json.dumps(P), '| folds', FOLDS, '| competition', COMP)

## Training molecules

In [ ]:
# Training molecules: structures of identity fold 0, which FPNet fold 0 never saw. Molecules with a timsTOF spectrum or
# an Enveda library (the hidden test is Enveda's timsTOF data) are taken first; the rest are a random fill.
import numpy as np
import pandas as pd
import pyarrow.dataset as pds
import pyarrow.parquet as pq

SEED, N_TESTLIKE, N_TOTAL = 0, 3000, 5000
names = set(pq.ParquetFile(COMP / 'train.parquet').schema_arrow.names)
st = pd.read_parquet(FOLDS, columns=['inchikey14', 'fold_identity']).drop_duplicates('inchikey14')
hold = set(st.inchikey14[st.fold_identity == 0])
cols = [c for c in ('inchikey14', 'normalized_smiles', 'instrument_type', 'ingest_lib') if c in names]
tr = pq.read_table(COMP / 'train.parquet', columns=cols).to_pandas()
tr = tr[tr.inchikey14.isin(hold)].reset_index(drop=True)
for c in ('instrument_type', 'ingest_lib'):
    if c in tr:
        print(c, tr[c].astype(str).value_counts().head(12).to_dict())
tr['tims'] = tr.instrument_type.astype(str).str.contains('tims', case=False) if 'instrument_type' in tr else False
tr['enveda'] = tr.ingest_lib.astype(str).str.contains('enveda', case=False) if 'ingest_lib' in tr else False
g = tr.groupby('inchikey14', sort=True).agg(smiles=('normalized_smiles', 'first'), tims=('tims', 'max'), enveda=('enveda', 'max'))
g['testlike'] = g.tims | g.enveda
rng = np.random.default_rng(SEED)
a_ = g.index[g.testlike].values
b_ = g.index[~g.testlike].values
pick = list(rng.choice(a_, min(N_TESTLIKE, len(a_)), replace=False))
pick += list(rng.choice(b_, min(N_TOTAL - len(pick), len(b_)), replace=False))
KEYS = sorted(set(pick))
del tr
print(f'fold-0 structures {len(g):,} (test-like {int(g.testlike.sum()):,}); picked {len(KEYS):,}')

spec_cols = [c for c in ('inchikey14', 'precursor_mz', 'ms2_mzs', 'ms2_normalized_intensities', 'adduct', 'ionization_mode',
                         'collision_energy_ev', 'instrument_type') if c in names]
sp = pds.dataset(str(COMP / 'train.parquet')).to_table(columns=spec_cols, filter=pds.field('inchikey14').isin(KEYS)).to_pandas()
SPECTRA = {}
for r in sp.to_dict('records'):
    SPECTRA.setdefault(r['inchikey14'], []).append(r)
for k, rs in SPECTRA.items():   # timsTOF spectra first, then the file order; at most MAX_SPECTRA as in the overlay
    rs.sort(key=lambda r: 0 if 'tims' in str(r.get('instrument_type', '')).lower() else 1)
    SPECTRA[k] = rs[:IR.MAX_SPECTRA]
del sp
print(f'spectra for {len(SPECTRA):,} molecules, {time.time() - T0:.0f} s')

## FPNet fold 0 logits

In [ ]:
# FPNet fold 0 logits for every training molecule (GPU), exactly as the overlay computes them on the test spectra.
Z = IR.fpnet_z(P, SPECTRA)
print(f'FPNet logits for {len(Z):,} molecules, {time.time() - T0:.0f} s')

## Candidates and features

In [ ]:
# Candidates of each molecule's own formula from the PubChem tier (at most IR.CAP, most documented first), their
# fingerprints in a process pool, then the stage-1 selection and the features. Label: the candidate's stored
# InChIKey-14 equals the molecule's.
import multiprocessing as mp

BUDGET_S = 5.0 * 3600
tier = IR.Tier(P)
nbits = len(np.load(P['bits']))
by_f = {}
for k in KEYS:
    if k not in Z:
        continue
    f0, m0 = IR.formula_mass(g.loc[k, 'smiles'])
    if f0 is None:
        continue
    by_f.setdefault(f0, [m0, []])[1].append(k)
formulas = sorted(by_f, key=lambda f: -len(by_f[f][1]))
print(f'{sum(len(v[1]) for v in by_f.values()):,} molecules over {len(formulas):,} formulas')
MOLS = []   # dicts: key, formula, testlike, n_window, in_window, X, y
stats = dict(no_window=0, too_few=0, done=0)
with mp.get_context('fork').Pool(os.cpu_count() or 2, initializer=IR.fp_init, initargs=(P['code'], P['bits'])) as pool:
    for b in range(0, len(formulas), 50):
        if time.time() - T0 > BUDGET_S:
            print('time budget reached at formula', b)
            break
        batch = formulas[b:b + 50]
        wins = {f: tier.window(by_f[f][0]) for f in batch}
        cache = {}
        IR.fingerprints(tier, {f: [c[0] for c in wins[f]] for f in batch}, pool, cache)
        for f in batch:
            cands = wins[f]
            for k in by_f[f][1]:
                if not cands:
                    stats['no_window'] += 1
                    continue
                mm = IR.molecule_matrix(cands, [cache.get((c[0], f)) for c in cands], Z[k], nbits)
                if mm is None:
                    stats['too_few'] += 1
                    continue
                sc, X, _ = mm
                y = np.array([c[1] == k for c in sc], np.int8)
                MOLS.append(dict(key=k, formula=f, testlike=bool(g.loc[k, 'testlike']), n_window=len(cands),
                                 in_window=any(c[1] == k for c in cands), X=X, y=y))
                stats['done'] += 1
        if (b // 50) % 10 == 0:
            print(f'  formulas {b + len(batch):,}/{len(formulas):,}, molecules {stats["done"]:,}, {time.time() - T0:.0f} s', flush=True)
print(stats, f'{time.time() - T0:.0f} s')

## LightGBM lambdarank

In [ ]:
# LightGBM lambdarank: five-fold out-of-fold scores (folds over molecules) for the report and the thresholds, then one
# model on every molecule with its answer among the candidates. Fixed rounds, deterministic, no early stopping.
import lightgbm as lgb

PARAMS = dict(objective='lambdarank', metric='ndcg', eval_at=[1, 5], learning_rate=0.05, num_leaves=31,
              min_data_in_leaf=50, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
              lambdarank_truncation_level=20, verbose=-1, seed=SEED, deterministic=True, num_threads=os.cpu_count() or 2)
NROUND = 400
F = IR.FEATS


def fit(idx):
    idx = [i for i in idx if MOLS[i]['y'].any()]
    X = np.concatenate([MOLS[i]['X'] for i in idx])
    y = np.concatenate([MOLS[i]['y'] for i in idx])
    return lgb.train(PARAMS, lgb.Dataset(X, y, group=[len(MOLS[i]['y']) for i in idx], feature_name=F), NROUND)


n = len(MOLS)
fold_of = np.random.default_rng(SEED).permutation(n) % 5
OOF = [None] * n
for k in range(5):
    m = fit([i for i in range(n) if fold_of[i] != k])
    for i in np.where(fold_of == k)[0]:
        OOF[i] = m.predict(MOLS[i]['X'])
    print(f'fold {k} done, {time.time() - T0:.0f} s', flush=True)
MODEL = fit(list(range(n)))

## Out-of-fold report and thresholds

In [ ]:
# Out-of-fold report: share of molecules whose answer is first by the joint ranker against the FPNet score, the
# channel score z(FPNet) + 0.25 x documentation and documentation alone; then the margins at which the joint ranker's
# first choice is right often enough to move a rank 1 (90%) or to take rank 2 (50%), fitted on the test-like molecules.
ci = {f: F.index(f) for f in F}
rows = []
for i, M in enumerate(MOLS):
    s, y, X = OOF[i], M['y'], M['X']
    o = np.argsort(-s, kind='stable')
    pos = np.where(y[o] == 1)[0]
    rows.append(dict(key=M['key'], testlike=M['testlike'], in_window=M['in_window'], in_sel=bool(y.any()), n_window=M['n_window'],
                     margin=float(s[o[0]] - s[o[1]]) if len(o) > 1 else 9.0, joint=bool(y[o[0]]),
                     joint_rank=int(pos[0]) + 1 if len(pos) else 0,
                     fz=bool(y[np.argmax(X[:, ci['fzz']])]), chan=bool(y[np.argmax(X[:, ci['chan']])]),
                     docs=bool(y[np.argmax(X[:, ci['pop']])])))
R = pd.DataFrame(rows)
R.to_parquet(WORK / 'isomer_ranker_oof.parquet', index=False)


def summary(d):
    rr = d.joint_rank.values
    return dict(n=len(d), in_window=round(d.in_window.mean(), 4), in_sel=round(d.in_sel.mean(), 4),
                top1_joint=round(d.joint.mean(), 4), top1_chan=round(d.chan.mean(), 4), top1_fz=round(d.fz.mean(), 4),
                top1_docs=round(d.docs.mean(), 4), mrr25_joint=round(float(np.where((rr > 0) & (rr <= 25), 1.0 / np.maximum(rr, 1), 0).mean()), 4))


def threshold(d, prec, min_n):
    o = np.argsort(-d.margin.values, kind='stable')
    c = np.cumsum(d.joint.values[o])
    k = np.arange(1, len(o) + 1)
    ok = np.where((c / k >= prec) & (k >= min_n))[0]
    if not len(ok):
        return None
    j = int(ok.max())
    return dict(margin=float(d.margin.values[o][j]), n=int(k[j]), share=round(float(k[j]) / len(d), 4), precision=round(float(c[j] / k[j]), 4))


cal = R[R.testlike] if int(R.testlike.sum()) >= 300 else R
TH = dict(swap=threshold(cal, 0.90, 30), insert=threshold(cal, 0.50, 50))
REPORT = dict(all=summary(R), testlike=summary(R[R.testlike]), other=summary(R[~R.testlike]),
              calibration_set='testlike' if cal is not R else 'all', thresholds=TH, stats=stats, seconds=round(time.time() - T0))
print(json.dumps(REPORT, indent=1))

## Outputs

In [ ]:
MODEL.save_model(str(WORK / 'isomer_ranker_model.txt'))
META = dict(features=F, thresholds={k: (v['margin'] if v else None) for k, v in TH.items()}, threshold_detail=TH,
            params={k: v for k, v in PARAMS.items() if k != 'num_threads'}, nround=NROUND, best_iteration=None,
            k1=[IR.K1_CHAN, IR.K1_FZ, IR.K1_POP], cap=IR.CAP, mass_tol=IR.MASS_TOL, max_spectra=IR.MAX_SPECTRA,
            molecules=len(MOLS), fitted_on=int(sum(M['y'].any() for M in MOLS)), seed=SEED, rdkit=rdBase.rdkitVersion,
            lightgbm=lgb.__version__, report=REPORT)
(WORK / 'isomer_ranker_meta.json').write_text(json.dumps(META, indent=1))
(WORK / 'isomer_ranker_report.json').write_text(json.dumps(REPORT, indent=1))
print(sorted(p.name for p in WORK.iterdir()))